# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
#loading the data
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
df_march_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS gsc_impressions,
           SUM(gsc_clicks) AS gsc_clicks,
           SUM(gsc_sum_position) AS gsc_sum_position,
           SUM(ga4_sessions) AS ga4_sessions,
           SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"March pages: {len(df_march_agg)}, April pages: {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March pages: 331437, April pages: 362172


In [27]:
df_april_agg.head()

,client_hash_id,content_hash_id,gsc_clicks_apr
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,1.0
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,0.0
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,0.0
3,client_62f4a7e64f5e0096,content_d47ba5533f9c8573,0.0
4,client_62f4a7e64f5e0096,content_50266f97d6233542,0.0


In [28]:
#last week's rule rewritten
import pandas as pd
import numpy as np

agg_df = df_march_agg.copy()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']
agg_df.loc[agg_df['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

signal_df = agg_df[agg_df['gsc_impressions'] > 0].copy()
signal_df['ctr'] = signal_df['gsc_clicks'] / signal_df['gsc_impressions']
signal_df['position_bucket'] = signal_df['gsc_avg_position'].apply(position_bucket)
position_avg_ctr = signal_df.groupby('position_bucket')['ctr'].mean().to_dict()

rule_df = agg_df.copy()
rule_df['ctr'] = rule_df['gsc_clicks'] / rule_df['gsc_impressions']
rule_df['engagement_rate'] = rule_df['ga4_engaged_sessions'] / rule_df['ga4_sessions']
rule_df['position_bucket'] = rule_df['gsc_avg_position'].apply(position_bucket)
rule_df['peer_avg_ctr'] = rule_df['position_bucket'].map(position_avg_ctr)

eligible = (rule_df['gsc_impressions'] >= 50) | (rule_df['ga4_sessions'] >= 10)
rule_df = rule_df[eligible].copy()

conditions = [
    (rule_df['gsc_impressions'] >= 50) & rule_df['ctr'].notna() & rule_df['peer_avg_ctr'].notna()
        & (rule_df['ctr'] < 0.5 * rule_df['peer_avg_ctr']),
    (rule_df['ga4_sessions'] >= 10) & rule_df['engagement_rate'].notna()
        & (rule_df['engagement_rate'] < 0.10),
]
rule_df['action'] = np.select(conditions, ['snippet_fix', 'content_fix'], default='monitor')
rule_df['reason_code'] = np.select(conditions,
    ['CTR_below_half_position_peers', 'engagement_below_10pct_reliable'], default='no_flag_triggered')
rule_df['rule_score'] = np.select(conditions,
    [(rule_df['peer_avg_ctr'] - rule_df['ctr']) * rule_df['gsc_impressions'],
     rule_df['ga4_sessions'] * (0.10 - rule_df['engagement_rate']) * 10], default=0)

print(f"March-eligible population: {len(rule_df)} pages")
print(rule_df['action'].value_counts())

March-eligible population: 116512 pages
action
snippet_fix    75663
monitor        28755
content_fix    12094
Name: count, dtype: int64


 **Rule**: decline_label = 1 if April clicks < 80% of March clicks

 Only counted if: (a) the page is still tracked in April at all, and

(b) it had at least 5 clicks in March (so the ratio isn't noise off tiny counts)

In [29]:
# April outcome
MIN_MARCH_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

merged = rule_df.merge(df_april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
tracked_in_april = merged['gsc_clicks_apr'].notna()
print(f"March pages with no April tracking at all (excluded, can't score a fair outcome): {(~tracked_in_april).sum()}")

labelable = merged[tracked_in_april & (merged['gsc_clicks'] >= MIN_MARCH_CLICKS_FOR_LABEL)].copy()
labelable['decline_label'] = (labelable['gsc_clicks_apr'] < DECLINE_THRESHOLD * labelable['gsc_clicks']).astype(int)

print(f"Excluded for < {MIN_MARCH_CLICKS_FOR_LABEL} March clicks: {(tracked_in_april & (merged['gsc_clicks'] < MIN_MARCH_CLICKS_FOR_LABEL)).sum()}")
print(f"Final labelable population: {len(labelable)} pages")
print(f"Base rate (decline_label == 1): {labelable['decline_label'].mean():.3f}")

March pages with no April tracking at all (excluded, can't score a fair outcome): 0
Excluded for < 5 March clicks: 87707
Final labelable population: 28805 pages
Base rate (decline_label == 1): 0.545


## 1. Method choice and why

**Lane:** Refresh / Content Opportunity Scoring.

I already have a target (`decline_label` — did the page's clicks fall in April vs. March) and features that were
all known before that decision point. That means I'm not in exploring-mode (no need for clustering or
correlation digging to find a target) — I already have one, so I go straight into supervised models.

**Order I'm using, and why:**
1. **The rule (from last week)** — kept exactly as-is, re-scored on this same population. This is my baseline to beat.
2. **Logistic regression** — my first learned model, because it's the simplest one that can use a target. It's
   readable (I can literally print what it's leaning on), so if it already beats the rule, I don't need anything fancier.
3. **Random forest** — added because some of these signals probably only matter together (e.g. low CTR only
   really means something when there's real impression volume behind it). That's exactly the kind of pattern a
   straight line struggles with, and where a forest earns its place.

**Skipped on purpose:** decision tree and gradient boosting, I'm not using them as
logistic regression already covers "simple and readable," and random forest already covers "catches interactions."
Adding more models on top wouldn't change my decision, just make the notebook longer.

In [30]:
FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

print(f"{len(FEATURES)} features going into logistic regression and random forest:")

8 features going into logistic regression and random forest:


## 2. Split design

I'm splitting by **client** — every `client_hash_id` on one side only, never split across train and test.

The real question I care about is "does this work for a client I haven't seen before?" If I split randomly instead,
the same client's pages could land in both train and test — and a model could just learn to recognize *that client*
("this smells like Client X, who declines about half the time") instead of learning what an actually-declining page
looks like. That's not a real skill, it's memorizing names, and grouping by client is the only way to stop it.

To show this isn't just a theory, I ran the same forest with a random split and a client-grouped split below —
same features, same data, only the split changes.

**Result:** Precision@10 went from 0.82 (random split) to 0.64 (grouped by client) — lower, same direction as
expected, but a smaller gap than a worst-case example would show. That's a fair result to report as-is: it doesn't
prove the forest is only memorizing clients, but it does confirm random split is the easier, less trustworthy exam.
Grouped-by-client is what I'm using for the real comparison below.


In [31]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# feature matrix
def build_X(df, feature_cols):
    X = df[feature_cols].copy()
    if 'gsc_avg_position' in X.columns:
        worst = X['gsc_avg_position'].max()
        X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    if 'ga4_sessions' in X.columns:
        X['ga4_sessions'] = X['ga4_sessions'].fillna(0)
    if 'ga4_engaged_sessions' in X.columns:
        X['ga4_engaged_sessions'] = X['ga4_engaged_sessions'].fillna(0)
    if 'engagement_rate' in X.columns:
        X['engagement_rate'] = X['engagement_rate'].fillna(0)
    if 'ctr' in X.columns:
        X['ctr'] = X['ctr'].fillna(0)
    if 'peer_avg_ctr' in X.columns:
        X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    X = X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))
    return X

# precision@k: sort by score, break exact ties using content_hash_id as a fixed second key
def precision_at_k(ids, scores, labels, k):
    temp = pd.DataFrame({'content_hash_id': ids, 'score': scores, 'label': labels})
    ranked = temp.sort_values(['score', 'content_hash_id'], ascending=[False, True])
    top_k = ranked.head(k)
    return top_k['label'].mean()

In [32]:
X_full = build_X(labelable, FEATURES)
y = labelable['decline_label'].values
groups = labelable['client_hash_id'].values

In [33]:
X_full.isna().sum()

,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
ga4_engaged_sessions,0
ctr,0
engagement_rate,0
peer_avg_ctr,0
pos_11-20,0
pos_21+,0


In [34]:
rf_model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)

kf = KFold(n_splits=5, shuffle=True, random_state=0)
random_p10 = [] #precision@10
for train, test in kf.split(X_full): #random split
    rf_model.fit(X_full.iloc[train], y[train])
    random_preds = rf_model.predict_proba(X_full.iloc[test])[:, 1]
    random_p10.append(precision_at_k(labelable['content_hash_id'].values[test], random_preds, y[test], 10))

gkf = GroupKFold(n_splits=5) #group split
grouped_p10 = [] #precision@10
for train, test in gkf.split(X_full, y, groups):
    assert len(set(groups[train]) & set(groups[test])) == 0, "client leakage across a fold"
    rf_model.fit(X_full.iloc[train], y[train])
    grp_preds = rf_model.predict_proba(X_full.iloc[test])[:, 1]
    grouped_p10.append(precision_at_k(labelable['content_hash_id'].values[test], grp_preds, y[test], 10))

In [35]:
print(f"Same forest, same features — Precision@10:")
print(f"  random split (client can appear on both sides): {np.mean(random_p10):.2f}  per fold: {[round(x,2) for x in random_p10]}")
print(f"  grouped split (every client on one side only):  {np.mean(grouped_p10):.2f}  per fold: {[round(x,2) for x in grouped_p10]}")

Same forest, same features — Precision@10:
  random split (client can appear on both sides): 0.80  per fold: [np.float64(0.8), np.float64(0.9), np.float64(0.7), np.float64(0.8), np.float64(0.8)]
  grouped split (every client on one side only):  0.56  per fold: [np.float64(0.7), np.float64(0.8), np.float64(0.5), np.float64(0.6), np.float64(0.2)]


## 3. Train + compare vs my baseline

Same March-eligible population as `w04`, same features, same 5 client-grouped folds (`GroupKFold` on `client_hash_id`,
zero-overlap checked every fold) — for the rule, logistic regression, and random forest alike. Primary metric is
Precision@50, with @10 and @100 as side checks.

**Result:**

| | Precision@10 | Precision@50 | Precision@100 |
|---|---|---|---|
| Rule (Week 4) | 0.480 | 0.472 | 0.498 |
| Logistic regression | 0.580 | **0.592** | 0.574 |
| Random forest | 0.600 | 0.572 | 0.556 |

Both logistic regression and random forest beat the rule in 4 of 5 client-held-out folds — a real, consistent
improvement over the baseline, not a fluke of averaging. Between the two models, results are close: random forest
is slightly ahead at Precision@10, logistic regression is slightly ahead at Precision@50 and @100. Since
Precision@50 is the metric this whole track has used since ML-03, **logistic regression is the one I'd recommend** —
it performs at least as well as the forest here, and it's the simpler, more explainable model. The forest's added
complexity isn't earning its keep on this data.

**Sensitivity check:** `decline_label` is built from clicks, and two features (`gsc_clicks`, `ctr`) are close
cousins of it. Removing both drops the random forest's Precision@50 from 0.572 to 0.520 — a real but moderate
decline, not a collapse. Both features are fair to keep (they're known before the decision point, same as the
rule uses `ctr` too), but this is worth naming honestly: part of the model's apparent lift is riding on
near-duplicates of the answer, not purely independent signal from position, sessions, or engagement.

In [36]:
k_values = [10,50,100]
results = []
rule_scores_all = labelable['rule_score'].values #rule predictions

for fold, (train,test) in enumerate(gkf.split(X_full,y,groups)):
  assert len(set(groups[train]) & set(groups[test])) == 0, "client leakage across a fold"

  scale = StandardScaler()
  X_train_scaled = scale.fit_transform(X_full.iloc[train])
  X_test_scaled = scale.transform(X_full.iloc[test])
  regression_model = LogisticRegression(max_iter=1000)
  regression_model.fit(X_train_scaled, y[train])
  preds = regression_model.predict_proba(X_test_scaled)[:,1] #logistic regression preds

  random_forest = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)
  random_forest.fit(X_full.iloc[train], y[train])
  rf_preds = random_forest.predict_proba(X_full.iloc[test])[:,1] #random forest preds

  test_ids = labelable['content_hash_id'].values[test]
  row = {'fold': fold, 'n_test': len(test), 'base_rate': round(y[test].mean(), 3)}
  for k in k_values:
      row[f'rule_p{k}'] = round(precision_at_k(test_ids, rule_scores_all[test], y[test], k), 3)
      row[f'logreg_p{k}'] = round(precision_at_k(test_ids, preds, y[test], k), 3)
      row[f'rf_p{k}'] = round(precision_at_k(test_ids, rf_preds, y[test], k), 3)
  results.append(row)




In [37]:
results_df = pd.DataFrame(results)
print("Per-fold results:")
print(results_df[['fold', 'n_test', 'base_rate', 'rule_p50', 'logreg_p50', 'rf_p50']].to_string(index=False))

print("\nAverages across folds:")
print(results_df[[c for c in results_df.columns if c.startswith(('rule_', 'logreg_', 'rf_'))]].mean().round(3))

Per-fold results:
 fold  n_test  base_rate  rule_p50  logreg_p50  rf_p50
    0    6799      0.514      0.40        0.68    0.74
    1    5502      0.733      0.66        0.78    0.84
    2    5502      0.483      0.36        0.52    0.56
    3    5502      0.455      0.36        0.56    0.40
    4    5500      0.546      0.58        0.42    0.26

Averages across folds:
rule_p10       0.480
logreg_p10     0.580
rf_p10         0.560
rule_p50       0.472
logreg_p50     0.592
rf_p50         0.560
rule_p100      0.498
logreg_p100    0.574
rf_p100        0.566
dtype: float64


In [38]:
logreg_wins = (results_df['logreg_p50'] > results_df['rule_p50']).sum()
logreg_losses = (results_df['logreg_p50'] < results_df['rule_p50']).sum()
print(f"Logistic regression vs rule at Precision@50, fold by fold: better in {logreg_wins}/5, worse in {logreg_losses}/5")

rf_wins = (results_df['rf_p50'] > results_df['rule_p50']).sum()
rf_losses = (results_df['rf_p50'] < results_df['rule_p50']).sum()
print(f"Random forest vs rule at Precision@50, fold by fold: better in {rf_wins}/5, worse in {rf_losses}/5")

Logistic regression vs rule at Precision@50, fold by fold: better in 4/5, worse in 1/5
Random forest vs rule at Precision@50, fold by fold: better in 4/5, worse in 1/5


In [39]:
FEATURES_NO_CLICKS = [f for f in FEATURES if f not in ['gsc_clicks', 'ctr']]
X_no_clicks = build_X(labelable, FEATURES_NO_CLICKS)

sens_results = []
for fold, (train, test) in enumerate(gkf.split(X_no_clicks, y, groups)):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=fold)
    rf.fit(X_no_clicks.iloc[train], y[train])
    s = rf.predict_proba(X_no_clicks.iloc[test])[:, 1]
    test_ids = labelable['content_hash_id'].values[test]
    sens_results.append(precision_at_k(test_ids, s, y[test], 50))

print(f"Random forest Precision@50 — full features (incl. clicks/CTR): {results_df['rf_p50'].mean():.3f}")
print(f"Random forest Precision@50 — clicks/CTR removed:              {sum(sens_results)/len(sens_results):.3f}")

Random forest Precision@50 — full features (incl. clicks/CTR): 0.560
Random forest Precision@50 — clicks/CTR removed:              0.516


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Interpretation:**

- **False positives** consistently share poor search position (13th–33rd) and very low engagement rate
  (under 13%). The model has learned this combination as "high risk," but several of these pages didn't
  actually decline — some even gained clicks. This suggests the model is partly confusing "chronically weak"
  with "actively declining" — a page can sit at a low, stable floor without getting worse.
- **False negatives** are the opposite pattern: strong pages (top-3 search position, hundreds of sessions,
  solid click volume) that the model scored as very low risk, but which still lost 26–49% of their clicks in
  April. A page looking healthy in March gives the model no early-warning signal at all here — this is a real
  blind spot, and arguably the costliest one, since these are higher-traffic pages than the ones the rule or
  model would otherwise flag first.
- Overall: the model is good at catching pages that already look weak, but not at anticipating a real drop in
  a currently-strong page. That's a limitation worth stating plainly rather than letting the headline Precision@50
  number imply the model "understands decline" in general.
- All of this is **decision-support**, not proof of cause: a high predicted-risk score means a page matches a
  pattern worth a human reviewer's time first, not that anything is confirmed broken.

In [40]:
# collect out-of-fold logistic regression predictions for every page (same folds as Section 3)
oof_probs = np.zeros(len(labelable))

for fold, (train, test) in enumerate(gkf.split(X_full, y, groups)):
    scale = StandardScaler()
    X_train_scaled = scale.fit_transform(X_full.iloc[train])
    X_test_scaled = scale.transform(X_full.iloc[test])
    regression_model = LogisticRegression(max_iter=1000)
    regression_model.fit(X_train_scaled, y[train])
    oof_probs[test] = regression_model.predict_proba(X_test_scaled)[:, 1]

labelable_scored = labelable.copy()
labelable_scored['logreg_prob'] = oof_probs

display_cols = ['gsc_impressions', 'gsc_clicks', 'gsc_clicks_apr', 'gsc_avg_position',
                'ga4_sessions', 'engagement_rate', 'logreg_prob', 'decline_label']

print("Confident but wrong (high predicted risk, page did NOT decline):")
false_pos = labelable_scored[(labelable_scored['logreg_prob'] > 0.7) & (labelable_scored['decline_label'] == 0)]
print(false_pos[display_cols].sort_values('logreg_prob', ascending=False).head(5).to_string(index=False))

print("\nMissed entirely (low predicted risk, page DID decline):")
false_neg = labelable_scored[(labelable_scored['logreg_prob'] < 0.2) & (labelable_scored['decline_label'] == 1)]
print(false_neg[display_cols].sort_values('logreg_prob').head(5).to_string(index=False))

Confident but wrong (high predicted risk, page did NOT decline):
 gsc_impressions  gsc_clicks  gsc_clicks_apr  gsc_avg_position  ga4_sessions  engagement_rate  logreg_prob  decline_label
        194579.0       242.0           319.0         32.786981        2603.0         0.013446     0.999286              0
        244931.0       669.0           612.0         15.173490         891.0         0.115600     0.994119              0
         83293.0        22.0            45.0         29.833095        1231.0         0.002437     0.968591              0
          6451.0        19.0            17.0         13.871803        1283.0         0.000779     0.957220              0
           431.0         6.0            14.0          8.531323        1043.0         0.021093     0.953793              0

Missed entirely (low predicted risk, page DID decline):
 gsc_impressions  gsc_clicks  gsc_clicks_apr  gsc_avg_position  ga4_sessions  engagement_rate  logreg_prob  decline_label
        154358.0      25

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.